[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-03-data-artifacts.ipynb#scrollTo=aa1bb2cc)

---
# Day 3 · Data Artifacts and the Metaflow Store
**certified-journeys / metaflow-certified** · Day 3 · Learn

> **Goal for today:** Understand how Metaflow versions every artifact by run ID, store rich objects (datasets, metrics, model configs) across steps, use the Client API to retrieve past artifacts, and explore the `.metaflow` directory structure on disk.

In [ ]:
%pip install -q metaflow

## Step 1 · The Metaflow Artifact Model

Every time you assign `self.attribute = value` inside a `@step`, Metaflow:
1. **Serializes** the value (pickle by default, custom codecs for NumPy/pandas)
2. **Content-addresses** the serialized bytes (SHA-256 hash)
3. **Stores** the bytes in the datastore keyed by `(flow_name, run_id, step_name, task_id, artifact_name)`
4. **Deduplicates** — identical objects share storage (important for large datasets)

This gives you:

| Property | Mechanism |
|---|---|
| Reproducibility | Any run ID → any step → any artifact, forever |
| Versioning | Each run has its own artifact namespace |
| Deduplication | Same data in two runs = stored once |
| Time-travel | `Flow('X')['42']['step'].task.data.attr` |
| Lazy loading | Artifacts are deserialized only when accessed |

The **local datastore** is `.metaflow/` in your working directory.
In production, it's an S3 bucket or GCS bucket — the Client API is identical.

In [ ]:
%%writefile artifact_demo_flow.py
from metaflow import FlowSpec, step
import json
import math
import random

class ArtifactDemoFlow(FlowSpec):
    """
    Demonstrates storing diverse artifact types:
    datasets, metrics dictionaries, model configs, and nested objects.
    """

    @step
    def start(self):
        random.seed(42)
        # Store a dataset as a list of dicts (simulates loading from a CSV)
        # Production equivalent: self.df = pd.read_csv('s3://bucket/data.csv')
        self.dataset = [
            {"id": i, "x": round(random.gauss(0, 1), 4), "y": round(random.gauss(5, 2), 4)}
            for i in range(200)
        ]
        self.dataset_meta = {
            "n_rows": len(self.dataset),
            "features": ["x", "y"],
            "source": "synthetic_gaussian"
        }
        print(f"Loaded dataset: {self.dataset_meta}")
        self.next(self.compute_stats)

    @step
    def compute_stats(self):
        # Compute descriptive statistics — store as a nested dict artifact
        def stats(values):
            n   = len(values)
            mu  = sum(values) / n
            var = sum((v - mu) ** 2 for v in values) / n
            return {
                "mean":   round(mu, 4),
                "std":    round(math.sqrt(var), 4),
                "min":    round(min(values), 4),
                "max":    round(max(values), 4),
                "median": round(sorted(values)[n // 2], 4)
            }

        x_vals = [row["x"] for row in self.dataset]
        y_vals = [row["y"] for row in self.dataset]

        # Nested dict artifact — Metaflow serializes it transparently
        self.stats = {
            "x": stats(x_vals),
            "y": stats(y_vals)
        }
        print(f"Stats for x: {self.stats['x']}")
        print(f"Stats for y: {self.stats['y']}")
        self.next(self.fit_model)

    @step
    def fit_model(self):
        # Fit a simple linear regression y = m*x + b using least squares
        # Production equivalent: sklearn Pipeline, XGBoost, or PyTorch model
        x_vals = [row["x"] for row in self.dataset]
        y_vals = [row["y"] for row in self.dataset]

        n     = len(x_vals)
        x_bar = sum(x_vals) / n
        y_bar = sum(y_vals) / n

        # Ordinary Least Squares formulas
        numerator   = sum((x - x_bar) * (y - y_bar) for x, y in zip(x_vals, y_vals))
        denominator = sum((x - x_bar) ** 2 for x in x_vals)
        slope     = numerator / denominator if denominator != 0 else 0
        intercept = y_bar - slope * x_bar

        # Store the model as a serializable dict — not a live object
        # This is a best practice: store parameters, not stateful objects when possible
        self.model = {
            "type": "linear_regression_ols",
            "slope":     round(slope, 6),
            "intercept": round(intercept, 6),
            "n_train":   n
        }
        print(f"Model fitted: y = {slope:.4f}*x + {intercept:.4f}")
        self.next(self.evaluate)

    @step
    def evaluate(self):
        # Evaluate with R² and MAE using the fitted model parameters
        x_vals = [row["x"] for row in self.dataset]
        y_vals = [row["y"] for row in self.dataset]

        slope     = self.model["slope"]
        intercept = self.model["intercept"]

        predictions = [slope * x + intercept for x in x_vals]
        residuals   = [y - yhat for y, yhat in zip(y_vals, predictions)]

        y_bar   = sum(y_vals) / len(y_vals)
        ss_res  = sum(r ** 2 for r in residuals)
        ss_tot  = sum((y - y_bar) ** 2 for y in y_vals)
        r2      = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        mae     = sum(abs(r) for r in residuals) / len(residuals)

        # Store metrics as a flat dict — easy to compare across runs
        self.metrics = {
            "r2":  round(r2, 4),
            "mae": round(mae, 4),
            "n":   len(y_vals)
        }
        print(f"Evaluation metrics: {self.metrics}")
        self.next(self.end)

    @step
    def end(self):
        print("\n" + "=" * 50)
        print("ArtifactDemoFlow complete")
        print(f"  Dataset rows:    {self.dataset_meta['n_rows']}")
        print(f"  Model type:      {self.model['type']}")
        print(f"  R²:              {self.metrics['r2']}")
        print(f"  MAE:             {self.metrics['mae']}")
        print("All artifacts versioned by run ID in .metaflow/")
        print("=" * 50)

if __name__ == '__main__':
    ArtifactDemoFlow()

**What just happened?**

- We defined a flow that stores four distinct artifact types: a **list of dicts** (dataset), a **metadata dict**, a **nested stats dict**, a **model parameters dict**, and a **flat metrics dict**.
- **All pure Python — no external dependencies** beyond Metaflow itself, so this runs on a fresh Colab.
- Storing model parameters as a `dict` rather than a live scikit-learn object is a best practice — dicts are more portable and survive Python version upgrades.
- The `dataset_meta` pattern (small summary dict alongside the full data) is useful for quick inspection via the Client API without loading the full dataset.

In [ ]:
# Run the artifact demo flow
!python artifact_demo_flow.py run --no-pylint 2>&1

**What just happened?**

- The pipeline ran all 5 steps and stored every `self.*` artifact to `.metaflow/`.
- The model fit a line to Gaussian noise — R² near 0 is expected (no true linear relationship in the data).
- **Every artifact is now accessible by run ID** — including `dataset` (200 rows), `stats`, `model`, and `metrics`.
- Notice Metaflow's timestamp log format: `[YYYY-MM-DD HH:MM:SS.mmm] FlowName/run_id/step/task_id`.

## Step 2 · Accessing Previous Run Artifacts with the Client API

The **Metaflow Client API** is a first-class interface for programmatic artifact retrieval.
It abstracts over the datastore — the same code works whether artifacts are on disk (local) or in S3 (production).

Key Client API objects:

```python
from metaflow import Flow, Run, Step, Task, DataArtifact

Flow('Name')                    # all runs of a flow
Flow('Name').latest_run         # most recent run
Flow('Name').latest_successful_run  # most recent successful run
Flow('Name')['run_id']          # specific run by ID (string)
run['step_name']                # a step within a run
step.task                       # the single task (for linear steps)
task.data.artifact_name         # deserialize an artifact
task.data                       # all artifacts as a namespace object
```

In [ ]:
from metaflow import Flow

# Get the latest run
flow = Flow("ArtifactDemoFlow")
run  = flow.latest_run
print(f"Run ID: {run.id}")
print(f"Successful: {run.successful}")
print(f"Created: {run.created_at}")
print()

# Retrieve the metrics artifact from the 'evaluate' step
evaluate_task = run["evaluate"].task
metrics = evaluate_task.data.metrics
print("Metrics artifact:")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print()

# Retrieve the model artifact
fit_task = run["fit_model"].task
model = fit_task.data.model
print("Model artifact:")
for k, v in model.items():
    print(f"  {k}: {v}")
print()

# Retrieve the stats artifact (nested dict)
stats_task = run["compute_stats"].task
stats = stats_task.data.stats
print("Stats artifact (x feature):")
for k, v in stats["x"].items():
    print(f"  x.{k}: {v}")

**What just happened?**

- `run['evaluate'].task` is shorthand when a step has exactly one task (linear flows always do).
- **Artifacts are deserialized lazily** — accessing `task.data.metrics` triggers pickle.loads only at that point.
- The nested `stats` dict was retrieved exactly as stored — Python dicts round-trip through pickle losslessly.
- **`run['fit_model']` is step-level access** — we don't need to reload the entire run to get one step's artifacts.

## Step 3 · Artifact Versioning by Run ID

Every Metaflow run gets a **unique integer run ID** assigned at the moment `python flow.py run` is called.
Run IDs are **monotonically increasing** within a flow — run 2 always came after run 1.

This means:
- You can always retrieve *exactly* the artifacts from a specific experiment
- You can compare any two runs by their IDs
- Deleting old runs is explicit — nothing is auto-expired

Let's run the flow a second time with a different seed to generate two versioned runs to compare.

In [ ]:
%%writefile artifact_demo_flow_v2.py
from metaflow import FlowSpec, step, Parameter
import math
import random

class ArtifactDemoFlow(FlowSpec):
    """Same flow with a Parameter so we can vary the seed from the CLI."""

    # Parameter makes CLI arguments first-class: python flow.py run --seed 99
    seed = Parameter('seed', help='Random seed for data generation', default=42, type=int)

    @step
    def start(self):
        random.seed(self.seed)
        self.dataset = [
            {"id": i, "x": round(random.gauss(0, 1), 4), "y": round(random.gauss(5, 2), 4)}
            for i in range(200)
        ]
        self.dataset_meta = {"n_rows": len(self.dataset), "seed": self.seed}
        print(f"Dataset loaded with seed={self.seed}, n={len(self.dataset)}")
        self.next(self.fit_model)

    @step
    def fit_model(self):
        x_vals = [r["x"] for r in self.dataset]
        y_vals = [r["y"] for r in self.dataset]
        n      = len(x_vals)
        x_bar  = sum(x_vals) / n
        y_bar  = sum(y_vals) / n
        num    = sum((x - x_bar) * (y - y_bar) for x, y in zip(x_vals, y_vals))
        den    = sum((x - x_bar) ** 2 for x in x_vals) or 1e-12
        slope     = num / den
        intercept = y_bar - slope * x_bar
        self.model = {"slope": round(slope, 6), "intercept": round(intercept, 6)}
        self.next(self.evaluate)

    @step
    def evaluate(self):
        x_vals = [r["x"] for r in self.dataset]
        y_vals = [r["y"] for r in self.dataset]
        preds  = [self.model["slope"] * x + self.model["intercept"] for x in x_vals]
        y_bar  = sum(y_vals) / len(y_vals)
        ss_res = sum((y - yhat) ** 2 for y, yhat in zip(y_vals, preds))
        ss_tot = sum((y - y_bar) ** 2 for y in y_vals) or 1e-12
        self.metrics = {
            "r2":  round(1 - ss_res / ss_tot, 4),
            "mae": round(sum(abs(y - yhat) for y, yhat in zip(y_vals, preds)) / len(y_vals), 4),
            "seed": self.seed
        }
        print(f"seed={self.seed} | R²={self.metrics['r2']} | MAE={self.metrics['mae']}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Run complete for seed={self.seed}")

if __name__ == '__main__':
    ArtifactDemoFlow()

In [ ]:
# Run twice with different seeds — each creates a separate versioned run
!python artifact_demo_flow_v2.py run --seed 42 --no-pylint 2>&1
print("\n" + "=" * 60 + "\n")
!python artifact_demo_flow_v2.py run --seed 99 --no-pylint 2>&1

**What just happened?**

- We ran the same flow twice with different seeds — **both runs are now stored** with different run IDs.
- `Parameter` is a Metaflow feature that makes CLI arguments first-class artifacts — `self.seed` is automatically stored alongside every other artifact.
- The two runs have slightly different R² and MAE because the datasets were generated with different random seeds.
- **Neither run overwrites the other** — Metaflow versioning is additive, not destructive.

In [ ]:
from metaflow import Flow

# Compare all runs of ArtifactDemoFlow side by side
# Note: this queries the local .metaflow datastore — same API works for S3 in production
flow = Flow("ArtifactDemoFlow")

print(f"{'Run ID':>8}  {'Seed':>6}  {'R²':>8}  {'MAE':>8}  {'OK':>5}")
print("-" * 45)

run_data = []
for run in flow.runs():
    if not run.successful:
        continue
    try:
        task = run["evaluate"].task
        m = task.data.metrics
        # Safely handle runs that might not have 'seed' in metrics
        seed = m.get("seed", "?")
        run_data.append((run.id, seed, m["r2"], m["mae"]))
    except Exception:
        pass

# Sort by run ID (ascending) to show chronological order
run_data.sort(key=lambda x: int(x[0]))

for run_id, seed, r2, mae in run_data:
    print(f"{run_id:>8}  {str(seed):>6}  {r2:>8.4f}  {mae:>8.4f}  {'✓':>5}")

if run_data:
    best = min(run_data, key=lambda x: x[3])  # best MAE
    print(f"\nBest run by MAE: run {best[0]} (seed={best[1]}, MAE={best[3]})")

**What just happened?**

- We queried **all stored runs** and compared their metrics in a table — no external MLflow needed.
- `run.successful` filters out failed or incomplete runs gracefully.
- **Run IDs are integers** stored as strings — sort with `int(x[0])` for correct ordering.
- This pattern scales: add more parameters, more metrics, more runs — the Client API always returns consistent objects.

## Step 4 · Exploring the .metaflow Directory Structure

When you run a flow locally, Metaflow creates a `.metaflow/` directory:

```
.metaflow/
  ArtifactDemoFlow/
    data/
      <sha256_prefix>/
        <sha256_hash>      ← content-addressed artifact bytes
    1/                     ← run ID
      _parameters/
        0/
          _task_ok         ← sentinel file: step succeeded
          data             ← artifact blob references
      start/
        0/                 ← task ID (0 for linear steps)
          _task_ok
          data
          ...
```

Key insight: **artifact data is stored once** in `.metaflow/FlowName/data/` by content hash.
Multiple runs referencing the same data share the same bytes — this is deduplication.

In production (S3), the structure is identical but under `s3://bucket/metaflow/`.

In [ ]:
import os

def walk_metaflow(root, max_depth=4, max_files=5):
    """Walk .metaflow directory with depth and file limits for readability."""
    if not os.path.exists(root):
        print(f"Directory not found: {root}")
        return
    print(f"Directory structure of: {root}/")
    print()
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath.replace(root, "").count(os.sep)
        if depth >= max_depth:
            dirnames.clear()  # stop descending
            continue
        indent = "  " * depth
        rel = os.path.relpath(dirpath, root)
        print(f"{indent}{'.' if rel == '.' else rel}/")
        # Show first max_files files at this level
        shown = filenames[:max_files]
        for f in shown:
            fpath = os.path.join(dirpath, f)
            size  = os.path.getsize(fpath)
            print(f"{indent}  {f}  ({size:,} bytes)")
        if len(filenames) > max_files:
            print(f"{indent}  ... and {len(filenames) - max_files} more files")

walk_metaflow(".metaflow")

In [ ]:
import os

# Measure total datastore size and count artifacts
total_size  = 0
total_files = 0
data_size   = 0
data_files  = 0

for dirpath, dirnames, filenames in os.walk(".metaflow"):
    for fname in filenames:
        fpath = os.path.join(dirpath, fname)
        size  = os.path.getsize(fpath)
        total_size  += size
        total_files += 1
        # The 'data' subdirectory holds serialized artifact bytes
        if "/data/" in dirpath or dirpath.endswith("/data"):
            data_size  += size
            data_files += 1

print(f"Total .metaflow directory:")
print(f"  Files:  {total_files}")
print(f"  Size:   {total_size:,} bytes ({total_size / 1024:.1f} KB)")
print()
print(f"Artifact data (serialized Python objects):")
print(f"  Files:  {data_files}")
print(f"  Size:   {data_size:,} bytes ({data_size / 1024:.1f} KB)")
print()
meta_size = total_size - data_size
print(f"Metadata (run/step/task records):")
print(f"  Size:   {meta_size:,} bytes ({meta_size / 1024:.1f} KB)")

**What just happened?**

- We walked the entire `.metaflow/` directory to understand the physical layout.
- **Content-addressed storage** means two runs with identical data share the same bytes — reducing storage for repeated experiments.
- The metadata files (`_task_ok`, `data` pointer files) are tiny — the bulk of storage is the serialized artifact bytes.
- **In production**, swap `.metaflow/` for `s3://your-bucket/metaflow/` by setting the `METAFLOW_DATASTORE_SYSROOT_S3` env var — the Client API is unchanged.

## Step 5 · Advanced Client API Patterns

Beyond simple `run['step'].task.data.attr`, the Client API supports:

| Pattern | Code |
|---|---|
| Latest successful run | `Flow('X').latest_successful_run` |
| Specific run by ID | `Flow('X')['42']` |
| All runs with filter | `[r for r in Flow('X').runs() if r.successful]` |
| Step with multiple tasks | `list(run['step'])` — fan-out steps have N tasks |
| Run tags | `run.tags` — metadata set with `--tag` at run time |
| Artifact as Python object | `task.data.attr` — deserialized and ready to use |
| List all artifact names | `[a.id for a in task]` |

Tags are especially useful for organizing experiments:
```bash
python flow.py run --tag experiment:baseline --tag version:v1
```

In [ ]:
from metaflow import Flow

flow = Flow("ArtifactDemoFlow")
run  = flow.latest_successful_run  # skip failed runs automatically

print(f"Latest successful run: {run.id}")
print(f"Tags: {run.tags}")
print()

# List all artifact names for a specific step
evaluate_task = run["evaluate"].task
print("Artifacts available on 'evaluate' task:")
for artifact in evaluate_task:
    # Each artifact object has .id (name) and .data (the value)
    try:
        val = artifact.data
        val_repr = repr(val)[:60] + "..." if len(repr(val)) > 60 else repr(val)
        print(f"  {artifact.id:20s}  {val_repr}")
    except Exception as e:
        print(f"  {artifact.id:20s}  <error: {e}>")

print()

# Use artifacts directly as Python objects — no deserialization boilerplate
metrics = evaluate_task.data.metrics
dataset = run["start"].task.data.dataset  # retrieve the full dataset
print(f"Dataset sample (first 3 rows):")
for row in dataset[:3]:
    print(f"  {row}")
print(f"\nMetrics: {metrics}")

**What just happened?**

- `for artifact in task` iterates over **all artifacts** stored on that task — useful for discovery.
- **`artifact.data` deserializes on demand** — iterating artifact names is cheap, loading all data is not.
- We retrieved the full 200-row `dataset` list from the `start` step — it round-tripped through pickle without any explicit serialization code.
- **`latest_successful_run`** is safer than `latest_run` in automated pipelines — it skips partial/failed runs.

In [ ]:
# Challenge: Write a flow called ExperimentTrackingFlow that:
# 1. In 'start': generates datasets with n_samples = 100, 200, and 500
#    Store them as self.small_data, self.medium_data, self.large_data
# 2. In 'profile': for each dataset, compute and store:
#    self.profiles = {'small': {mean, std, n}, 'medium': {...}, 'large': {...}}
# 3. In 'report': print a formatted comparison table
# 4. In 'end': print the dataset size that had the highest std
#
# After running, use the Client API to:
#   - Retrieve self.profiles from the 'profile' step
#   - Print which dataset size had the highest standard deviation

# Your solution:
%%writefile experiment_tracking_flow.py
from metaflow import FlowSpec, step
import random
import math

class ExperimentTrackingFlow(FlowSpec):

    @step
    def start(self):
        random.seed(42)
        # TODO: generate three datasets of sizes 100, 200, 500
        # Each dataset is a list of floats: random.gauss(0, 1)
        # Store as self.small_data, self.medium_data, self.large_data
        ...
        self.next(self.profile)

    @step
    def profile(self):
        # TODO: compute mean and std for each dataset
        # Store as self.profiles = {'small': {'n': ..., 'mean': ..., 'std': ...}, ...}
        ...
        self.next(self.report)

    @step
    def report(self):
        # TODO: print a table comparing the three profiles
        ...
        self.next(self.end)

    @step
    def end(self):
        # TODO: print which dataset had the highest std
        ...

if __name__ == '__main__':
    ExperimentTrackingFlow()

# Run: !python experiment_tracking_flow.py run --no-pylint
# Then retrieve self.profiles via the Client API:
# from metaflow import Flow
# run = Flow('ExperimentTrackingFlow').latest_run
# profiles = run['profile'].task.data.profiles
# print(profiles)

---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| Content-addressing | Artifacts stored by SHA-256 hash — identical objects deduplicated |
| Run ID versioning | Every run gets a unique integer ID; old runs are never overwritten |
| Client API | `Flow('X').latest_run` → `run['step'].task.data.attr` — works local and S3 |
| `task.data.attr` | Lazy deserialization — only loads the artifact you ask for |
| `for artifact in task` | Enumerate all artifact names and values on a task |
| `.metaflow/` structure | `data/` = content-addressed bytes; `run_id/step/task/` = metadata |
| `Parameter` | CLI argument that becomes a versioned artifact — great for experiment tracking |
| `latest_successful_run` | Safe accessor for production — skips failed/partial runs |

> **Tip:** Any Python object assigned to `self` in a step becomes a versioned artifact — Metaflow serializes it automatically.

---
## What's next
**Day 4** → Parameters and Flow Configuration — use `Parameter` to make flows configurable from the CLI, set defaults, add type validation, and build configurable ML pipelines.

Mark Day 3 complete in your [tracker](../index.html).